# Setup

In [ ]:
import h5py
import numpy as np

import data

def pack_data(h5_file, key_selector, met_data):
    data_groups = [(key, group) for (key, group) in h5_file.items() if key_selector in key]
    feat_dict = {}
    for (key, group) in data_groups:
        arrays = [np.reshape(arr, shape) for (arr, shape) in zip(group["data"], group["shape"])]
        try:
            data = np.stack(arrays, 0)
        except Exception:
            raise ValueError(f"Data cannot be combined -- likely different shapes.")
        feat_dict[key] = data
        specimen_ids = h5_file["specimens"][group["sp_idx"][:]].astype(str)
    classes = met_data.get_specimens(specimen_ids, outputs = ["class"])["class"]
    met_data.close()
    stacked_feats = separate_arbors(feat_dict, classes)
    return {"data": stacked_feats.astype("float32"), "specimens": np.char.encode(specimen_ids)}

def separate_arbors(feat_dict, classes):
    arbor_types = {"exc": {"apical": 2, "basal": 3}, "inh": {"axon": 0, "basal": 1}}
    container = None
    for (feat_key, feat_arr) in feat_dict.items():
        if container is None:
            container = np.zeros(feat_arr.shape + (4,))
        arbor_name = feat_key.split("_")[0]
        for (neuron_class, arbor_maps) in arbor_types.items():
            if arbor_name in arbor_maps:
                container_idx = arbor_maps[arbor_name]
                container[classes == neuron_class, ..., container_idx] = feat_arr[classes == neuron_class]
    return container

# Run

In [ ]:
source_file = "/Users/ian.convy/code/arbors/outputs/patchseq_sholl.hdf5"
output_file = ""
data_key = "sholl"
data_name = "sholl"

data_paths = {"patchseq": "../data/patchseq.hdf5",
              "EM": "../data/EM.hdf5",
              "smartseq": "../data/smartseq.hdf5",
              "10x": "../data/10x.hdf5",
              "trunc": "../data/patch_smart_trunc.hdf5"}
data_keys = {
    "logcpm": [["trunc", "logcpm"], ["10x", "logcpm_aligned"]],
    "pca-ipfx": [["patchseq", "pca-ipfx"]],
    "arbors": [["patchseq", "arbors"], ["EM", "arbors"]]}
met_data = data.MET_Data(data_paths, **data_keys)

source_file = h5py.File(source_file, "r")
pack_dict = pack_data(source_file, data_key, met_data)
output_file = h5py.File(output_file, "a")
group = output_file.create_group(data_name)
for (key, arr) in pack_dict.items():
    group.create_dataset(key, data = arr)